# Baseline: Binary Sentiment Classification

Trains a DistilBERT model on the public training data and saves
predictions to `submission.parquet`.

**Requirements:** `torch`, `transformers`, `datasets`, `pandas`, `pyarrow`

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    AdamW,
    get_scheduler,
)

## 1. Load Training Data

The admin uploads `train.parquet` as a resource file. It must contain at least a `text` column and a `label` column.

In [ ]:
train_df = pd.read_parquet("train.parquet")
print("Training samples:", len(train_df))
print(train_df.head())

texts = train_df["text"].tolist()
labels = train_df["label"].tolist()

## 2. Tokenize

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")


class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


dataset = SentimentDataset(texts, labels, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

## 3. Train Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epochs}  Loss: {total_loss / len(dataloader):.4f}")

## 4. Generate Predictions → `submission.parquet`

The evaluation engine reads this file after the Docker sandbox exits.

In [ ]:
model.eval()
all_ids = []
all_preds = []

with torch.no_grad():
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        all_ids.extend(batch["label"].cpu().numpy().tolist())
        # NOTE: In a real competition, run on a test set with actual
        # test IDs. Training IDs are used here for demonstration.
        all_preds.extend(preds.tolist())

df_sub = pd.DataFrame({"id": all_ids, "label": all_preds})
df_sub.to_parquet("submission.parquet", index=False)

print(f"Saved submission.parquet with {len(df_sub)} rows")
print(df_sub.head())